# 02 — Pandas fundamentals


In [ ]:
from pathlib import Path

ROOT = Path('..').resolve()
RAW = ROOT / 'data' / 'raw'
OUT = ROOT / 'data' / 'output'
OUT.mkdir(parents=True, exist_ok=True)


In [ ]:
## Задание 5.1

## Создайте DataFrame из словаря (имитация CSV): {'user_id': [1, 2, 3, 4, 5], 'status': ['active', 'inactive', 'active', 'banned', 'active'], 'revenue': [150, 0, 320, 50, 210]}.
## Проверьте типы данных через df.info(). Приведите user_id к str, status к category.
## Отфильтруйте строки, где status == 'active' и revenue > 200.
## Выберите из результата только колонки user_id и revenue.
import pandas as pd, numpy as np

df = pd.DataFrame({
    'user_id': [1, 2, 3, 4, 5],
    'status': ['active', 'inactive','active', 'banned', 'active'],
    'revenue': [150, 0, 320, 50, 210]
    })

df.info()
df['user_id'] = df["user_id"].astype("str")
df["status"] = df["status"].astype("category")
print("\n", df)

df = df[(df['status'] == 'active') & (df['revenue'] > 200)]
print("\n", df[['user_id','revenue']])

In [ ]:
## Задание 5.2

## Создайте DataFrame 5x3 с индексом ['A','B','C','D','E'] и колонками ['x','y','z']. Заполните числами 1–15.
## С помощью .loc замените значение в строке 'C', колонке 'y' на 999.
## С помощью .iloc замените последнюю строку всех колонок на 0.
## Создайте подтаблицу только с колонками x и z для строк ['B','D']. 
## Измените в ней колонку z на -1 так, чтобы не вызвать предупреждение и не затронуть исходный DataFrame.
import pandas as pd, numpy as np

df = pd.DataFrame(data=np.reshape(np.arange(1, 16), (5, 3)),index=['A','B','C','D','E'], columns=['x','y','z'])

df.loc['C', 'y'] = 999

df.iloc[4] = 0

df_2 = df.loc[['B','D'],['x', 'z']].copy()
df_2.loc[:, 'z'] = -1
print(df_2)

In [ ]:
## Задание 5.3
## Цель: Применить загрузку, типы, даты, фильтрацию и экспорт в связном пайплайне.

## 📋 Условие:
## Сгенерируйте DataFrame с колонками:
    ## order_id: строки вида 'ORD-0001', 'ORD-0002', ...
    ## date_str: строки формата 'DD.MM.YYYY' (например, '15.03.2024')
    ## amount: числа с пропусками (NaN) и отрицательными значениями
    ## region: строки ['EU', 'US', 'APAC']
## Преобразуйте date_str в datetime и отфильтруйте заказы только за 2024 год.
## Замените NaN в amount на 0. Удалите строки, где amount < 0.
## Добавьте колонку tier: 'High' если amount > 5000, иначе 'Low' (используйте np.where).
## Сохраните результат в файл clean_orders.csv без индекса.
import pandas as pd, numpy as np
datatypes = {'order_id': 'str', 'date_str': 'str', 'amount': 'float64', 'region': 'category'}

np.random.seed(0)
n = 1000
raw = pd.DataFrame(
    {
    'order_id': [f'ORD-{i:04d}' for i in range(n)],
    'date_str': pd.date_range('2023-01-01', periods=n, freq='D').strftime('%d.%m.%Y'),
    'amount': np.random.normal(3000, 1500, n),
    'region': np.random.choice(['EU', 'US', 'APAC'], n)
    }).astype(datatypes)

raw.loc[np.random.choice(n, 50), 'amount'] = np.nan
raw.loc[np.random.choice(n, 20), 'amount'] = -np.random.uniform(100, 500, 20)

raw['date'] = pd.to_datetime(raw['date_str'], format = '%d.%m.%Y')
df_dates = raw[raw['date'].dt.year == 2024].copy()

df_dates['amount'] = df_dates['amount'].fillna(0)
df_cleaner = df_dates[df_dates['amount'] >= 0].copy()

df_cleaner['tier'] = np.where(df_cleaner['amount'] > 5000, 'High', 'Low')
df_cleaner.to_csv(str(OUT / 'clean_orders.csv'), index=False)

In [ ]:
## Задание 6.1
## Цель: Отработать groupby + agg с одной и несколькими функциями.

## 📋 Условие:
## Создайте DataFrame с колонками: category (['A','B','C']), product (['X','Y']), sales (числа 10–100).
## Посчитайте сумму продаж по каждой категории (одна функция).
## Посчитайте сумму и среднее продаж по комбинации category + product (две функции).
## Оформите результат через именованные агрегации: total_sales и avg_sales.
import pandas as pd, numpy as np

np.random.seed(42)

df = pd.DataFrame({
    'category': np.random.choice(['A', 'B', 'C'], size=12),
    'product':  np.random.choice(['X', 'Y'], size=12),
    'sales':    np.random.randint(10, 101, size=12)
})

df.groupby("category")['sales'].sum()
df.groupby(["category", "product"]).agg(total_sales=("sales", "sum"), avg_sales=("sales", "mean"))


In [ ]:
## Задание 6.2
## Цель: Научиться считать несколько бизнес-показателей на группу, именовывать колонки, фильтровать группы и сбрасывать индекс.

## 📋 Условие:
## Создайте DataFrame с колонками: user_id (строки), region (строки), order_amount (float), is_completed (bool: True/False).
## Сгруппируйте данные по region.
## Рассчитайте для каждого региона:
    ## total_revenue: сумма order_amount
    ## avg_order: среднее order_amount
    ## completed_orders: количество True в is_completed (подсказка: sum для bool считает True как 1)
    ## total_orders: общее число строк в группе
## Отфильтруйте регионы, где total_orders >= 50.
## Сбросьте индекс, чтобы region стал обычной колонкой.
## ✅ Разрешено: groupby, agg (named syntax), reset_index, булева фильтрация >=, sum/mean/count как строки.
## ❌ Запрещено: Циклы, .apply(), ручное суммирование, transform в этом задании.
## 📝 Ожидаемый вывод: DataFrame с колонками ['region', 'total_revenue', 'avg_order', 'completed_orders', 'total_orders'], отсортированный по region (опционально).
import pandas as pd, numpy as np

np.random.seed(42)

df = pd.DataFrame({
    'user_id': np.arange(1, 201),
    'region': np.random.choice(['EU', 'NA', 'SEA', 'ASIA'], size=200),
    'order_amount': np.random.uniform(100, 1001, size=200),
    'is_completed': np.random.choice([True, False], size=200)
})

groupped = df.groupby('region').agg(total_revenue=("order_amount", "sum"), avg_order=("order_amount", "mean"), completed_orders=('is_completed', 'sum'), total_orders=("user_id","count")).round({'total_revenue': 1, 'avg_order': 1}).copy()
filtered = groupped[groupped['total_orders'] >= 50].reset_index().copy()
filtered.sort_values('region', ascending=False)

In [ ]:
## Задание 6.3
## Цель: Освоить transform для расчёта относительных метрик без изменения формы таблицы.

## 📋 Условие:
## Создайте DataFrame с колонками: user_id, category, monthly_spend.
## С помощью transform добавьте колонку category_avg_spend: среднее monthly_spend внутри каждой category.
## Рассчитайте колонку spend_share: доля трат пользователя от среднего по его категории (monthly_spend / category_avg_spend).
## Добавьте булеву колонку is_high_value: True, если monthly_spend > category_avg_spend * 1.5, иначе False.
## Выведите первые 10 строк итогового DataFrame.
## ✅ Разрешено: groupby, transform('mean'), арифметические операторы /, *, >, присваивание новых колонок.
## ❌ Запрещено: agg, merge, циклы, apply, ручное вычисление среднего по группе.
## 📝 Ожидаемый вывод: DataFrame с исходными колонками + ['category_avg_spend', 'spend_share', 'is_high_value']
import pandas as pd, numpy as np
n = 1000
np.random.seed(42)
df = pd.DataFrame({
    'user_id': np.arange(1, n+1),
    'category': np.random.choice(['Groceries', 'Cosmetics', 'Clothes', 'Supplements'], size=n),
    'monthly_spend': np.random.randint(100, 100000, size=n)
})

df["category_avg_spend"] = df.groupby('category')["monthly_spend"].transform('mean')
df['spend_share'] = (df['monthly_spend'] / df['category_avg_spend'])
df['is_high_value'] = df['monthly_spend'] > (df['category_avg_spend'] * 1.5)
print(df[:10])

In [ ]:
## Задание 7.1
## Цель: Научиться объединять таблицы с валидацией, отслеживать потери строк и обрабатывать пропуски после склейки.

## 📋 Условие:
## Создайте df_users с колонками: user_id (1..100), tier ('Basic'/'Pro').
## Создайте df_orders с колонками: order_id (1..300), user_id (случайные 1..90, т.е. 10 пользователей без заказов), amount.
## Выполните left merge пользователей с заказами. Обязательно проверьте целостность ключей через validate='1:m'.
## Заполните пропуски в amount нулями (это пользователи без заказов).
## Посчитайте общую выручку по tier.
## ✅ Разрешено: merge, validate, fillna, groupby, agg, rename (из 🛠️).
## ❌ Запрещено: apply, циклы, ручное сопоставление по индексам.
## 📝 Ожидаемый результат: DataFrame с колонками ['tier', 'total_revenue'].
    
import pandas as pd, numpy as np

df_users = pd.DataFrame({
    'user_id': np.arange(1, 101),
    'tier': np.random.choice(['Basic', 'Pro'], size=100)
})

df_orders = pd.DataFrame({
    'order_id': np.arange(1, 301),
    'user_id': np.random.randint(1, 91, size = 300),
    'amount': np.random.randint(1, 1001, size = 300)
})

res = pd.merge(df_users, df_orders, on='user_id', how='left', validate='1:m')
res['amount'] = res['amount'].fillna(0)

print(pd.DataFrame(res.groupby('tier')['amount'].sum()).rename(columns={"amount": "total_revenue"}))

In [ ]:
## Задание 7.2
## Цель: Научиться собирать данные из нескольких источников и разворачивать их в аналитический вид.

## 📋 Условие:
## Создайте 3 DataFrame: q1, q2, q3. У каждого колонки: product, region, sales. Заполните случайными данными.
## Склейте их вертикально в один df_full, сбросив индекс.
## Добавьте колонку quarter(значения 'Q1', 'Q2', 'Q3')на основе порядка склейки или вручную перед объединением.
## Создайте pivot_table: в индексе region, в колонках quarter, значения sales, агрегация sum.
## Заполните пропуски нулями.
## Добавьте колонку growth_q3_vs_q2: разница между Q3 и Q2 для каждого региона.
## ✅ Разрешено: concat, pivot_table, арифметика колонок, rename (из 🛠️).
## ❌ Запрещено: merge, melt, циклы, apply.
## 📝 Ожидаемый результат: DataFrame с регионами в индексе, колонками Q1, Q2, Q3 и growth_q3_vs_q2.
    
import pandas as pd, numpy as np
n=100
np.random.seed(42)
q1 = pd.DataFrame({
    'product': np.random.choice(['Fish', 'Meat', 'Vegetables', 'Fruits', 'Milk'], size=n),
    'region': np.random.choice(['EU', 'NA', 'SA', 'SEA'], size=n),
    'sales': np.random.randint(100, 100000, size=n)
})

q2 = pd.DataFrame({
    'product': np.random.choice([ 'Vegetables', 'Fruits', 'Yogurt'], size=n),
    'region': np.random.choice(['EU', 'NA', 'SA', 'SEA'], size=n),
    'sales': np.random.randint(100, 100000, size=n)
})

q3 = pd.DataFrame({
    'product': np.random.choice([ 'Vegetables', 'Bananas', 'Meat', 'Fish'], size=n),
    'region': np.random.choice(['NA', 'SA', 'SEA'], size=n),
    'sales': np.random.randint(100, 100000, size=n)
})
q1['quarter'], q2['quarter'], q3['quarter'] = 'Q1', 'Q2', 'Q3'
df_full = pd.concat([q1, q2, q3], ignore_index=True)
# print(f'\n{df_full[20:25]}\n')

pivot_table = pd.pivot_table(
    df_full,
    index='region',
    columns='quarter',
    values='sales',
    aggfunc='sum',
    fill_value=0
)
pivot_table['growth_q3_vs_q2'] = pivot_table['Q3'] - pivot_table['Q2']
print(pivot_table)

In [ ]:
# Задание 7.3
# Цель: Освоить melt для подготовки данных к долгосрочному анализу и расчёту долей.
    
# 📋 Условие:
## Создайте df_wide с колонками: city, jan_rev, feb_rev, mar_rev.
## Преобразуйте таблицу в long-формат через melt. Якорь: city. Значения: все месячные колонки. Назовите новую колонку с месяцами 'month', с выручкой 'revenue'.
## Посчитайте долю выручки каждого месяца от общей выручки города. (Подсказка: используйте groupby + transform из Блока 6).
## Отфильтруйте строки, где доля > 0.35.
## ✅ Разрешено: melt, groupby, transform, булева фильтрация, rename (из 🛠️).
## ❌ Запрещено: pivot, merge, apply, циклы.
## 📝 Ожидаемый результат: DataFrame с колонками ['city', 'month', 'revenue', 'share'], отфильтрованный по условию.
import pandas as pd, numpy as np
n=10
df_wide = pd.DataFrame({
    'city': np.random.choice(['Moscow', 'Kursk', 'Tula'], size = n),
    'jan_rev': np.random.randint(100, 100000, size=n),
    'feb_rev': np.random.randint(100, 100000, size=n),
    'mar_rev': np.random.randint(100, 100000, size=n)
})

df_wide = df_wide.melt(
    id_vars='city',
    value_vars=['jan_rev', 'feb_rev', 'mar_rev'],
    var_name='month',
    value_name='revenue'
)
group_rev = df_wide.groupby('city')['revenue'].transform('sum').copy()
df_wide['share'] = (df_wide['revenue'] / group_rev).round(2)
df_wide = df_wide[df_wide['share'] > 0.35]